In [2]:
import os

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sqlite3 import connect
from pandas import DataFrame

from Constants import Constants as const

In [3]:
tqdm.pandas(desc="Processing")

# Load latest NLRB data

In [9]:
nlrb_con = connect(os.path.join(const.DATABASE_PATH, 'union', 'NLRB', 'nlrb.db'))
# 1. 从 election 表中读取所有所需变量
election_raw = pd.read_sql("""
    SELECT election_id, case_number, date, tally_type
    FROM election
""", nlrb_con, parse_dates=['date'])

# 根据 case_number 和 tally_type 选择“最终有效结果”
def select_final_election(df):
    # 优先选择 run-off，其次 rerun，其次 initial
    for t_type in ['runoff', 'rerun', 'initial']:
        sub = df[df['tally_type'].str.lower() == t_type]
        if not sub.empty:
            return sub.sort_values('date').iloc[-1]  # 选择最后一次
    return df.sort_values('date').iloc[-1]  # 万一都没匹配上，退而求其次

election_df = election_raw.groupby('case_number', group_keys=False).apply(select_final_election).reset_index(drop=True)


# 2. 获取 filing 表数据
filing_df = pd.read_sql("""
    SELECT case_number, name, case_type, city, state
    FROM filing
""", nlrb_con)


# 3. 获取 tally 表数据
tally_df = pd.read_sql("""
    SELECT election_id, option, votes
    FROM tally
""", nlrb_con)

# 替换 votes 中的缺失值为 0
tally_df['votes'] = tally_df['votes'].fillna(0)

C:\Users\wangy\AppData\Local\Temp\ipykernel_24964\2532917584.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  election_df = election_raw.groupby('case_number', group_keys=False).apply(select_final_election).reset_index(drop=True)


In [4]:
# 每个 election_id 中票数最多的作为 vote_for，其余的作为 vote_against
def summarize_votes(group):
    votes_by_option = group.groupby('option')['votes'].sum()
    vote_for = votes_by_option.drop(labels='No union', errors='ignore').max() if not votes_by_option.drop(labels='No union', errors='ignore').empty else 0
    vote_against = votes_by_option.drop(labels='No union', errors='ignore').sum() - vote_for
    vote_no_union = votes_by_option.get('No union', 0)
    vote_against += vote_no_union
    return pd.Series({'vote_for': vote_for, 'vote_against': vote_against})

vote_summary_df = tally_df.groupby('election_id').progress_apply(summarize_votes).reset_index()


  0%|          | 0/32291 [00:00<?, ?it/s]

In [11]:
# 4. 合并所有数据
merged_df = election_df.merge(filing_df, on='case_number', how='left') \
                       .merge(vote_summary_df, on='election_id', how='left')

# 重命名变量
merged_df = merged_df.rename(columns={
    'case_number': 'casenumber',
    'name': 'employer',
    'case_type': 'type',
    'date': 'election_date',
})

merged_df['number_of_votes'] = merged_df['vote_for'] + merged_df['vote_against']

In [16]:
merged_df['election_date'].describe()

count                            30504
mean     2015-12-02 06:13:41.400471808
min                1994-03-04 00:00:00
25%                2011-01-07 00:00:00
50%                2015-07-22 12:00:00
75%                2020-12-14 00:00:00
max                2025-06-18 00:00:00
Name: election_date, dtype: object

In [17]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))

In [3]:
merged_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))

## add address information

In [15]:
from rapidfuzz import process, fuzz  # 更快且推荐使用 rapidfuzz
import warnings
warnings.filterwarnings('ignore')

conn = connect(os.path.join(const.DATABASE_PATH, 'union', 'NLRB', 'nlrb.db'))

# 7. 获取 participant 表中的信息
participant_df = pd.read_sql("""
    SELECT case_number, participant, type, subtype, city,
           address_1, address_2, state, zip
    FROM participant
""", conn)

conn.close()

# 保留 type/subtype 中与 employer 相关的行
employer_participants = participant_df[
    participant_df['subtype'].str.lower().str.contains('employer', na=False)
].copy()

# 建立雇主地址参考表
employer_address_map = {}
for idx, row in tqdm(merged_df.iterrows()):
    case = row['casenumber']
    emp_name = row['employer']

    # 在相同 case_number 内匹配最相近的 participant
    subset = employer_participants[(employer_participants['case_number'] == case) & employer_participants['participant'].notnull()]
    if subset.shape[0] == 1:
        match_row = subset.iloc[0]
        employer_address_map[case] = {
            'employercity': match_row.get('city'),
            'employeraddress1': match_row.get('address_1'),
            'employeraddress2': match_row.get('address_2'),
            'employerstate': match_row.get('state'),
            'employerzip': match_row.get('zip'),
        }
    elif not subset.empty:
        # fuzzy match
        match, score, match_idx = process.extractOne(emp_name, subset['participant'], scorer=fuzz.token_sort_ratio)

        if score >= 80:  # 匹配阈值可调
            match_row = subset.loc[match_idx]
            employer_address_map[case] = {
                'employercity': match_row.get('city'),
                'employeraddress1': match_row.get('address_1'),
                'employeraddress2': match_row.get('address_2'),
                'employerstate': match_row.get('state'),
                'employerzip': match_row.get('zip'),
            }

# 转换为 DataFrame 并合并
employer_address_df = pd.DataFrame.from_dict(employer_address_map, orient='index').reset_index()
employer_address_df = employer_address_df.rename(columns={'index': 'casenumber'})

# 合并到 merged_df
merged_df = merged_df.merge(employer_address_df, on='casenumber', how='left')

# 示例输出
print(merged_df[['casenumber', 'employer', 'employeraddress1', 'employerstate']].head())


0it [00:00, ?it/s]

     casenumber                               employer     employeraddress1  \
0  01-RC-020966  New England Mechanical Services of MA        Washington St   
1  01-RC-020986          Courtyard Nursing Care Center    200 Governors Ave   
2  01-RC-021152                        Carney Hospital  2100 Dorchester Ave   
3  01-RC-021561                        Telcom USA Inc.       309 Andover St   
4  01-RC-022002  Regional Transportation Program, Inc.    127 Saint John St   

  employerstate  
0            MA  
1            MA  
2            MA  
3            MA  
4            ME  


In [17]:
merged_df.head()

,election_id,casenumber,election_date,tally_type,employer,type,city,state,vote_for,vote_against,number_of_votes,employercity,employeraddress1,employeraddress2,employerstate,employerzip
0,1,01-RC-020966,1999-03-17,Initial,New England Mechanical Services of MA,RC,Woburn,MA,0.0,0.0,0.0,Boston,Washington St,None,MA,-
1,2,01-RC-020986,1999-04-22,Initial,Courtyard Nursing Care Center,RC,Medford,MA,101.0,47.0,148.0,Medford,200 Governors Ave,None,MA,02155-1644
2,3,01-RC-021152,2000-04-27,Initial,Carney Hospital,RC,Boston,MA,0.0,229.0,229.0,Dorchester,2100 Dorchester Ave,None,MA,02124-5615
3,4,01-RC-021561,2002-11-07,Initial,Telcom USA Inc.,RC,Lawrence,MA,47.0,52.0,99.0,Lawrence,309 Andover St,None,MA,01843-2227
4,5,01-RC-022002,2007-04-25,Initial,"Regional Transportation Program, Inc.",RC,Portland,ME,5.0,4.0,9.0,Portland,127 Saint John St,None,ME,04102-3042


In [19]:
merged_df.loc[:, 'employercity'] = merged_df['employercity'].fillna(merged_df['city'])
merged_df.loc[:, 'employerstate'] = merged_df['employerstate'].fillna(merged_df['state'])


In [20]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))


## merge union election data with knepper data

In [3]:
ue_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))
knepper_df = pd.read_stata(r'D:\OneDrive\Projects\UnionizationCSR\Data\Knepper_2018\merged_elections_1961_2009.dta')

C:\Users\Dell\AppData\Local\Temp\ipykernel_22880\1429232100.py:2: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  knepper_df = pd.read_stata(r'D:\OneDrive\Projects\UnionizationCSR\Data\Knepper_2018\merged_elections_1961_2009.dta')


In [7]:
ue_df['month_electionheld'] = ue_df['election_date'].dt.month
ue_df['electionhelddate'] = ue_df['election_date']
ue_df['year_electionheld'] = ue_df['election_date'].dt.year
ue_df['day_electionheld'] = ue_df['election_date'].dt.day
ue_df.rename(columns={'vote_for': 'votes_for', 'vote_against': 'votes_against'}, inplace=True)

In [6]:
ue_df.keys()

Index(['election_id', 'casenumber', 'election_date', 'tally_type', 'employer',
       'type', 'city', 'state', 'vote_for', 'vote_against', 'number_of_votes',
       'employercity', 'employeraddress1', 'employeraddress2', 'employerstate',
       'employerzip'],
      dtype='object')

In [8]:
common_keys = ['employer', 'employercity', 'employerstate', 'casenumber', 'month_electionheld','year_electionheld', 'votes_for', 'votes_against','type', 'day_electionheld', 'employeraddress1', 'employeraddress2', 'employerzip']

In [16]:
knepper_df['year_electionheld'].describe()

count    251051.000000
mean       1979.848369
std          12.540486
min        1921.000000
25%        1970.000000
50%        1977.000000
75%        1989.000000
max        2025.000000
Name: year_electionheld, dtype: float64

In [10]:
ue_df['year_electionheld'].describe()

count    30504.000000
mean      2015.418339
std          5.564471
min       1994.000000
25%       2011.000000
50%       2015.000000
75%       2020.000000
max       2025.000000
Name: year_electionheld, dtype: float64

In [12]:
knepper_df.loc[knepper_df['casenumber'].notnull() & knepper_df['casenumber'].duplicated()]

,employer,employercity,employerstate,casenumber,unit,election_type,incumbant,month_electionheld,year_electionheld,ind_code,...,incumbent,electioncity,electionstate,noofsustainedchallenges,unitloccity,unitlocstate,unitloccounty,employeraddress2,employerzip,number_of_votes
295,CAROLINA LUMBER CO A CORP,HUNTINGTON,WV,09-RM-00269,T,R,0.0,12.0,1961,,...,,,,NaN,,,,,,NaN
702,CONTINENTAL CONST CO,CORVALLIS,OR,36-RM-00248,C,,0.0,12.0,1961,,...,,,,NaN,,,,,,8.0
710,CROWN COACH CO,JOPLIN,MO,17-RC-03713,W,S,0.0,12.0,1961,,...,,,,NaN,,,,,,18.0
782,COOPER STUART F CO,LOS ANGELES,CA,21-RC-07114,D,B,0.0,12.0,1961,,...,,,,NaN,,,,,,17.0
941,CAROLINA LUMBER CO A CORP,HUNTINGTON,WV,09-RM-00269,T,R,0.0,12.0,1961,,...,,,,NaN,,,,,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250993,United Cerebral Palsy of Northeastern Maine,Bangor,ME,01-RC-22306,Healt,,NaN,4.0,2009,,...,*,Bangor,ME,0.0,Bangor,ME,PISCATAQUIS,,4401,15.0
251000,"Knight Protective Service, Inc.",Capitol Heights,MD,05-RC-16309,Guard,,NaN,6.0,2009,,...,,Washington,DC,0.0,Washington,DC,DIST OF COLUMBIA,,20743,9.0
251009,University of Chicago,Chicago,IL,13-RC-21815,Dept,,NaN,1.0,2009,,...,,Chicago,IL,0.0,Chicago,IL,COOK,,60637,23.0
251030,"Laro Service Systems, Inc.",Bay Shore,NY,29-RC-11713,Dept,,NaN,3.0,2009,,...,*,Flushing,NY,0.0,Bayshore,NY,QUEENS,271 Skip Lane,11706,19.0


In [15]:
knepper_df[knepper_df['casenumber'] == '01-RC-06694']

,employer,employercity,employerstate,casenumber,unit,election_type,incumbant,month_electionheld,year_electionheld,ind_code,...,incumbent,electioncity,electionstate,noofsustainedchallenges,unitloccity,unitlocstate,unitloccounty,employeraddress2,employerzip,number_of_votes
2478,DIAMON NATL CORP,CHELSEA,MA,01-RC-06694,A,R,0.0,1.0,1962,,...,,,,NaN,,,,,,4.0
4936,DIAMON NATL CORP,CHELSEA,MA,01-RC-06694,A,R,0.0,1.0,1962,,...,,,,NaN,,,,,,5.0


In [17]:
# 1. knepper_df 预处理

# 补零
knepper_df['votes_for'] = knepper_df['votes_for'].fillna(0)
knepper_df['votes_against'] = knepper_df['votes_against'].fillna(0)

# 标记是否有 casenumber
has_case = knepper_df['casenumber'].notna()

# 用于去重排序
knepper_df['total_votes'] = knepper_df['votes_for'] + knepper_df['votes_against']
knepper_df['date'] = pd.to_datetime(dict(year=knepper_df['year_electionheld'],
                                         month=knepper_df['month_electionheld'],
                                         day=knepper_df['day_electionheld']), errors='coerce')

# 有 casenumber 的按 casenumber 去重
knepper_with_case = knepper_df[has_case].sort_values(['casenumber', 'date', 'total_votes'], ascending=[True, False, False]) \
                                        .drop_duplicates(subset='casenumber', keep='first')

# 没有 casenumber 的，按 employer + type + 地址 + 日期匹配
dedup_cols = ['employer', 'type', 'employercity', 'employerstate', 'employeraddress1', 'employerzip', 'date']
knepper_without_case = knepper_df[~has_case].sort_values(['date', 'total_votes'], ascending=[False, False]) \
                                            .drop_duplicates(subset=dedup_cols, keep='first')

# 合并两部分
knepper_cleaned = pd.concat([knepper_with_case, knepper_without_case], ignore_index=True)

# 2. 只保留 common_keys
knepper_cleaned = knepper_cleaned[common_keys]
ue_df = ue_df[common_keys]

# 3. 限制年份范围
ue_df = ue_df[(ue_df['year_electionheld'] >= 1994) & (ue_df['year_electionheld'] <= 2025)]
knepper_cleaned = knepper_cleaned[(knepper_cleaned['year_electionheld'] >= 1961) & (knepper_cleaned['year_electionheld'] <= 2009)]

# 4. 合并两个 DataFrame，以 ue_df 为主（优先保留）
# 首先移除 knepper 中重复的 casenumber
knepper_nocasematch = knepper_cleaned[~knepper_cleaned['casenumber'].isin(ue_df['casenumber'])]

# 最终合并
final_union_df = pd.concat([ue_df, knepper_nocasematch], ignore_index=True)

# 排序（可选）
final_union_df = final_union_df.sort_values(['year_electionheld', 'month_electionheld', 'day_electionheld'])

# 输出查看
print(final_union_df.head())

                        employer employercity employerstate   casenumber  \
30538  SUPER MARKET DISTRIBUTORS  SPRINGFIELD            MA  01-RC-06331   
30542  DERAN CONFECTIONERY CO IN    CAMBRIDGE            MA  01-RC-06474   
30544  DERAN CONFECTIONERY CO IN    CAMBRIDGE            MA  01-RC-06517   
30545  AMERICAN DISTRICT TELEGRA       BOSTON            MA  01-RC-06531   
30546  AINSLEY CORP TEINER ENG D       MALDEN            MA  01-RC-06535   

       month_electionheld  year_electionheld  votes_for  votes_against type  \
30538                12.0               1961       20.0           19.0   RC   
30542                12.0               1961       15.0           33.0   RC   
30544                12.0               1961       78.0          221.0   RC   
30545                12.0               1961       10.0           37.0   RC   
30546                12.0               1961       19.0           19.0   RC   

       day_electionheld employeraddress1 employeraddress2 employerzi

In [19]:
final_union_df[const.NUM_VOTES] = final_union_df['votes_for'] + final_union_df['votes_against']
final_union_df[[const.NUM_VOTES, 'votes_for', 'votes_against']].describe()

,number_of_votes,votes_for,votes_against
count,245413.000000,245413.000000,245413.000000
mean,55.092678,27.880530,27.212149
std,151.074135,78.751256,85.382117
min,0.000000,0.000000,0.000000
25%,9.000000,4.000000,2.000000
50%,20.000000,10.000000,8.000000
75%,50.000000,25.000000,23.000000
max,32279.000000,18844.000000,13435.000000


In [20]:
final_union_df.to_pickle(os.path.join(const.TEMP_PATH, '1961_2024_union_election_data.pkl'))

## fuzzy match firm names

In [21]:
from name_matching.name_matcher import NameMatcher

In [36]:
ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))
final_union_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1961_2024_union_election_data.pkl'))

# Step 0: 清洗数据
final_union_df['employer'] = final_union_df['employer'].astype(str).str.lower()
ctat_df['conml'] = ctat_df['conml'].astype(str).str.strip().str.lower()

# 保存最终匹配结果
match_records = []

# 2. 构造每年匹配：t 和 t-1 年度的 Compustat 公司名
for year in sorted(final_union_df['year_electionheld'].dropna().unique()):
    print('Start to handle year: ', year)
    comp_subset = ctat_df[
        ctat_df['fyear'].isin([year, year - 1])
    ][['gvkey', 'conml']].drop_duplicates(subset='conml').copy()

    # 如果为空，跳过
    if compustat_subset.empty:
        continue

    # 工会选举当年的所有记录
    union_subset = final_union_df[final_union_df['year_electionheld'] == year].copy()
    union_subset = union_subset.drop_duplicates(subset='employer')

    # 准备 name matcher
    nm = NameMatcher(lowercase=True,
                     punctuations=True,
                     remove_ascii=True,
                     legal_suffixes=False,
                     common_words=False,
                     verbose=True)

    nm.set_distance_metrics(['discounted_levenshtein',
                                  'SSK',
                                  'fuzzy_wuzzy_token_sort'])
    nm.load_and_process_master_data('conml', comp_subset)
    matched_names = nm.match_names(to_be_matched=union_subset, column_matching='employer')
    for i in matched_names.index:
        match_row = matched_names.loc[i]
        if match_row['score'] >= 85:
            idx = matched_names.loc[i, 'match_index']  # 取原始索引
            employer_name = match_row['original_name']
            matched_conml = match_row['match_name']
            score = match_row['score']

            match_records.append({
                'index': i,
                'employer': employer_name,
                'matched_conml': matched_conml,
                'match_score': score,
                'gvkey': comp_subset.loc[idx, const.GVKEY],
                'match_year': year
            })

C:\Users\Dell\AppData\Local\Temp\ipykernel_21012\525319285.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))


Start to handle year:  1961
preprocessing...

preprocessing complete 
 searching for matches...



100%|██████████| 1/1 [00:00<00:00, 15.00it/s]


possible matches found   
 fuzzy matching...



100%|██████████| 1644/1644 [01:25<00:00, 19.21it/s]


...

done
Start to handle year:  2025
preprocessing...

preprocessing complete 
 searching for matches...



100%|██████████| 1/1 [00:00<00:00,  6.19it/s]

possible matches found   
 fuzzy matching...




100%|██████████| 648/648 [01:07<00:00,  9.54it/s]

done


In [37]:
match_df = pd.DataFrame(match_records)
match_df.head()

,index,employer,matched_conml,match_score,gvkey,match_year
0,45540,coca cola bottling co,cocacola bottling co of ny,85.302002,3142,1961
1,177717,raytheon co,raytheon co,100.000000,8972,1961
2,63566,dennison manufacturing co,dennison manufacturing co,100.000000,3868,1961
3,130296,wmca inc,mca inc,88.161505,6861,1961
4,39498,carrier corp,carrier corp,100.000000,2789,1961


In [46]:
match_df[match_df['employer'] == 'square d co']

,index,employer,matched_conml,match_score,gvkey,match_year
9,199033,square d co,square d co,100.0,9965,1961
59,199034,square d co,square d co,100.0,9965,1962
615,199037,square d co,square d co,100.0,9965,1965
880,199038,square d co,square d co,100.0,9965,1966
1342,199039,square d co,square d co,100.0,9965,1967
1675,199040,square d co,square d co,100.0,9965,1968
2388,199042,square d co,square d co,100.0,9965,1970
2888,199043,square d co,square d co,100.0,9965,1971
3528,199045,square d co,square d co,100.0,9965,1973
5108,199048,square d co,square d co,100.0,9965,1976


In [45]:
match_df[match_df['employer'].duplicated()]

,index,employer,matched_conml,match_score,gvkey,match_year
59,199034,square d co,square d co,100.000000,9965,1962
74,90934,general motors corp,general motors co,93.620340,5073,1962
76,90430,general foods corp,general foods corp,100.000000,5055,1962
78,177310,rath packing co the,rath packing co,86.560547,8953,1962
90,160419,otis elevator co,otis elevator co,100.000000,8198,1962
...,...,...,...,...,...,...
11231,118095,kinder morgan inc,kinder morgan inc,100.000000,6310,2025
11232,277122,kerberos international inc,mercer international inc,85.171935,14563,2025
11234,543685,centerpoint energy resources corp,centerpoint energy resources corp,100.000000,145350,2025
11235,579126,phillips 66,phillips 66,100.000000,170841,2025


In [39]:
final_union_df.head()

,employer,employercity,employerstate,casenumber,month_electionheld,year_electionheld,votes_for,votes_against,type,day_electionheld,employeraddress1,employeraddress2,employerzip,number_of_votes
30538,super market distributors,SPRINGFIELD,MA,01-RC-06331,12.0,1961,20.0,19.0,RC,NaN,,,,39.0
30542,deran confectionery co in,CAMBRIDGE,MA,01-RC-06474,12.0,1961,15.0,33.0,RC,NaN,,,,48.0
30544,deran confectionery co in,CAMBRIDGE,MA,01-RC-06517,12.0,1961,78.0,221.0,RC,NaN,,,,299.0
30545,american district telegra,BOSTON,MA,01-RC-06531,12.0,1961,10.0,37.0,RC,NaN,,,,47.0
30546,ainsley corp teiner eng d,MALDEN,MA,01-RC-06535,12.0,1961,19.0,19.0,RC,NaN,,,,38.0


In [43]:
final_union_df.loc[579126]

KeyError: 579126

In [51]:
# Step 3: 合并匹配结果
match_df = pd.DataFrame(match_records).rename(columns={'match_year': const.ELECTION_YEAR}).drop_duplicates(subset=[const.ELECTION_YEAR, 'employer'], keep='first')
ue_gvkey_df = final_union_df.drop(['gvkey'], axis=1).merge(match_df[[const.ELECTION_YEAR, 'employer', const.GVKEY]], how='left', on=[const.ELECTION_YEAR, 'employer'])
final_union_df.shape[0], ue_gvkey_df.shape[0]


(245413, 245413)

In [53]:
print(ue_gvkey_df[const.GVKEY].notnull().sum())
ue_gvkey_df[const.GVKEY] = ue_gvkey_df.groupby('employer')[const.GVKEY].ffill()
ue_gvkey_df[const.GVKEY] = ue_gvkey_df.groupby('employer')[const.GVKEY].bfill()
print(ue_gvkey_df[const.GVKEY].notnull().sum())

12360
13807


In [54]:
ue_gvkey_df.to_pickle(os.path.join(const.TEMP_PATH, '1950_2025_union_election_gvkey.pkl'))

### merge

In [6]:
ue_gvkey_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1950_2025_union_election_gvkey.pkl'))
previous_df = pd.read_csv(r'D:\Onedrive\Temp\Projects\UnionDisclosure\regression_data\NLRB_with_GVKEY.csv',
                          usecols=['casenumber', const.GVKEY])
ue_gvkey_df2 = ue_gvkey_df.merge(previous_df.drop_duplicates(subset=['casenumber']), how='left', on='casenumber')
ue_gvkey_df2[const.GVKEY] = ue_gvkey_df2['gvkey_x'].fillna((ue_gvkey_df2['gvkey_y']))
ue_gvkey_df2 = ue_gvkey_df2.drop(['gvkey_x', 'gvkey_y'], axis=1)
print(ue_gvkey_df2.shape[0], ue_gvkey_df2[const.GVKEY].notnull().sum())

245413 17151


In [15]:
previous_df.drop_duplicates(subset=['casenumber']).shape

(3949, 2)

In [8]:
ue_gvkey_df2[const.GVKEY] = ue_gvkey_df2.groupby('employer')[const.GVKEY].ffill()
ue_gvkey_df2[const.GVKEY] = ue_gvkey_df2.groupby('employer')[const.GVKEY].bfill()
print(ue_gvkey_df2[const.GVKEY].notnull().sum())

20455


In [12]:
ue_gvkey_df2.to_pickle(os.path.join(const.TEMP_PATH, '1950_2025_union_election_gvkey2.pkl'))

In [13]:
ue_gvkey_df2.loc[ue_gvkey_df2[const.ELECTION_YEAR].apply(lambda x: 2008 <= x)].describe()

,month_electionheld,year_electionheld,votes_for,votes_against,day_electionheld,number_of_votes,gvkey
count,31161.000000,31161.000000,31161.000000,31161.000000,31161.000000,31161.000000,3280.000000
mean,6.450146,2015.371009,29.380540,20.741215,15.682841,50.121755,50800.327439
std,3.356095,5.450583,129.420866,95.207242,8.577972,217.331651,61452.722407
min,1.000000,2008.000000,0.000000,0.000000,1.000000,0.000000,1019.000000
25%,4.000000,2010.000000,5.000000,1.000000,8.000000,8.000000,6557.000000
50%,6.000000,2015.000000,11.000000,6.000000,16.000000,19.000000,14477.000000
75%,9.000000,2020.000000,28.000000,19.000000,23.000000,48.000000,66065.000000
max,12.000000,2025.000000,18844.000000,13435.000000,31.000000,32279.000000,265188.000000


In [27]:
previous_df_all = pd.read_csv(r'D:\Onedrive\Temp\Projects\UnionDisclosure\regression_data\NLRB_with_GVKEY.csv',
                              parse_dates=['election_date']).drop_duplicates(subset=['casenumber'])
previous_df_all[const.ELECTION_YEAR] = previous_df_all['election_date'].dt.year
previous_df_all[const.ELECTION_MONTH] = previous_df_all['election_date'].dt.month
previous_df_all[const.ELECTION_DAY] = previous_df_all['election_date'].dt.day
previous_df_all[const.NUM_VOTES] = previous_df_all[const.NUM_VOTE_FOR] + previous_df_all[const.NUM_VOTE_AGAINST]
common_keys = set(ue_gvkey_df2.keys()).intersection(set(previous_df_all.keys()))
print(common_keys)

{'employer', 'year_electionheld', 'gvkey', 'type', 'day_electionheld', 'votes_against', 'number_of_votes', 'month_electionheld', 'casenumber', 'votes_for'}


In [31]:
previous_df_all.loc[previous_df_all[const.ELECTION_YEAR] >= 2008].shape

(1328, 17)

In [29]:
ue_gvkey_df3 = pd.concat([ue_gvkey_df2,
                         previous_df_all.loc[
                             ~previous_df_all['casenumber'].isin(ue_gvkey_df2['casenumber']),
                             list(common_keys)]], ignore_index=True)

In [33]:
ue_gvkey_df3.to_pickle(os.path.join(const.TEMP_PATH, '1950_2025_union_election_gvkey3.pkl'))